# Operaciones de aprendizaje automático
## Maestría en Inteligencia Artificial Aplicada

### TC5061 | Proyecto MiBici: propuesta y base reproducible | Colaborativo

#### **Equipo:** 7
#### **Nombre:** Luis Arturo Sánchez Davila
#### **Matrícula:** A01840576

## Objetivo
Notebook de **limpieza, partición y preprocesamiento** del proyecto MiBici (Paso 3). Convierte los hallazgos del EDA (`0.02-lasd-eda-p3`) en reglas explícitas, separa los datos en conjuntos temporales y deja listas las tablas para el modelado de P1, P2 y P3.

## Entradas y salidas
- **Entrada:** `data/interim/viajes_AAAA-MM_AAAA-MM.parquet` (`dataset.construir`) y la nomenclatura vigente.
- **Salidas en `data/processed/`:**
  - `viajes_limpios_AAAA-MM_AAAA-MM.parquet`: resultado de `dataset.limpiar` (reglas comunes).
  - `viajes_modelado_AAAA-MM_AAAA-MM.parquet`: viajes limpios con las columnas de modelado y la partición temporal. Es la entrada de `0.04-lasd-baseline-p3`.
  - `estaciones_train.parquet` y `demanda_estacion_hora.parquet`: tablas de preprocesamiento.

## Nota de reproducibilidad
El notebook se ejecuta desde la carpeta `notebooks/`. Las cifras corresponden a la corrida del 15-oct-2026 con los 81 meses del rango.

## Serie de notebooks de P3
| Notebook | Contenido | Entrega |
|---|---|---|
| `0.01-lasd-exploracion-carga` | Paso 1: carga y descripción | Dataset normalizado en `data/interim/` |
| `0.02-lasd-eda-p3` | Paso 2: análisis exploratorio | Hallazgos de calidad, tiempo y estaciones |
| `0.03-lasd-limpieza-p3` | Paso 3: limpieza, partición y preprocesamiento | `data/processed/` (viajes de modelado, estaciones y demanda) |
| `0.04-lasd-baseline-p3` | Paso 4 (4.1 a 4.4): modelo base y registro en MLflow | Modelo base y métricas (0.05 lo recalcula con el mismo código y semilla) |
| `0.05-lasd-mejoramodelo-p3` | Paso 4 (4.5 a 4.9): modelo alternativo y registro en MLflow | Comparación contra el baseline |

In [ ]:
# Importación de librerías y registro de versiones (evidencia de reproducibilidad)
from pathlib import Path
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("pyarrow     :", pyarrow.__version__)
print("matplotlib  :", matplotlib.__version__)

## Configuración y entradas

Las rutas y el rango salen de `tc5061_proyecto_equipo07/config.py`. Del dataset sin limpiar solo se necesita el número de registros (para la bitácora); la limpieza la ejecuta `dataset.limpiar`, y la partición y las tablas de preprocesamiento, `dataset.p3_preparar_modelado` y `features.py`.

La lógica reutilizable vive en el módulo `tc5061_proyecto_equipo07` (funciones `p3_*`); este notebook la llama y documenta los hallazgos.

In [ ]:
# Configuración compartida (el notebook se ejecuta desde notebooks/)
try:
    from tc5061_proyecto_equipo07 import dataset
except ModuleNotFoundError:  # paquete sin instalar; `make requirements` lo instala con `-e .`
    sys.path.insert(0, str(Path.cwd().parent))
    from tc5061_proyecto_equipo07 import dataset
from tc5061_proyecto_equipo07 import features, plots
from tc5061_proyecto_equipo07.config import (
    MES_FIN,
    MES_INICIO,
    P3_COLUMNAS_MODELADO,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    RUTA_DEMANDA_ESTACION_HORA,
    RUTA_ESTACIONES_TRAIN,
    RUTA_VIAJES_INTERIM,
    RUTA_VIAJES_LIMPIOS,
)

# Opciones de visualización de pandas
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
pd.set_option("display.max_colwidth", None)  # no recortar textos largos

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
RUTA_VIAJES_MODELADO = dataset.p3_ruta_viajes_modelado(MES_INICIO, MES_FIN)

# Registros del dataset sin limpiar, tomados de los metadatos del Parquet (sin cargarlo)
if not RUTA_VIAJES_INTERIM.exists():
    raise FileNotFoundError(f"Falta {RUTA_VIAJES_INTERIM.name}; ejecutar `make data` (ver 0.01-lasd)")
registros_originales = pq.ParquetFile(RUTA_VIAJES_INTERIM).metadata.num_rows

# Nomenclatura de estaciones (misma detección de codificación que los viajes)
nomenclatura = dataset.p3_cargar_estaciones()
print(f"Registros sin limpiar: {registros_originales:,} | estaciones en la nomenclatura: {len(nomenclatura)}")

## Paso 3: Limpieza, partición y preprocesamiento

En este paso se convierten los hallazgos del EDA en reglas explícitas, se separan los datos en conjuntos temporales y se dejan listas las tablas para el modelado. Se sigue el orden del pipeline (`references/pipeline-ml.md`):
1. **Limpieza** (3.1): cada regla se justifica con una cifra del EDA (`0.02-lasd-eda-p3`). La aplica `dataset.limpiar`, que registra cuántos registros retira cada una; el notebook la verifica.
2. **Partición** (3.2): temporal, con una brecha de 24 horas entre conjuntos para que ninguna etiqueta use datos del conjunto siguiente.
3. **Preprocesamiento** (3.3): tabla de estaciones calculada solo con entrenamiento y tabla de demanda por estación-hora, la base de P1 y P2.

**Principio:** nada de lo que se calcule con validación o prueba puede influir en una decisión de limpieza o de preprocesamiento.

### 3.1 Decisiones de limpieza

| Hallazgo (sección 2.2 de `0.02-lasd-eda-p3`) | Decisión | Justificación |
|---|---|---|
| 758 filas completas repetidas | **Eliminar** (conservar una) | Son idénticas en todas las columnas; no hay conflictos que resolver |
| 3,961 viajes con origen o destino sin nomenclatura | **Excluir** | Sin capacidad ni ubicación no se pueden usar en P2 ni P3; son el 0.015 % |
| 55,035 viajes con duración cero | **Excluir** | Probable desbloqueo fallido: no es un traslado. Su sensibilidad se medirá al modelar |
| 159 viajes de más de 24 h (máximo 469 h) | **Excluir** | La hora de llegada no es real; distorsionaría el balance neto de P2 |
| Viajes de 1 a 24 h | **Conservar** | El 99.9 % dura menos de 59 min; la cola moderada es plausible y no afecta los conteos |
| Nulos en `genero` (0.51 %) y `anio_de_nacimiento` (0.13 %) | **No imputar; descartar las columnas** | Ninguna pregunta las usa y son datos personales (minimización) |
| `usuario_id` | **Descartar** | Dato personal y las preguntas son por estación, no por persona |
| Años de nacimiento imposibles | **Sin acción** | Se descartan junto con la columna |
| Meses sin datos (ninguno con los 81 meses publicados) | **No imputar; excluir esas horas de la tabla de demanda** | Rellenar con ceros inventaría una demanda nula que no ocurrió |
| Horas 1:00 a 4:59 sin servicio | **Excluir de la tabla de demanda** | Son horas cerradas, no demanda cero medida |
| 4 estaciones con capacidad 0 | **Conservar los viajes; marcar la estación** | Sirven para P1; se excluyen solo de P2 |

In [ ]:
# Limpieza común: la aplica dataset.limpiar y deja el resultado en data/processed/.
# El log de esa función (abajo) es la bitácora: cuántos registros retira cada regla, en orden.
dataset.limpiar(MES_INICIO, MES_FIN)

# Para el modelado se leen solo las columnas que usan las tres preguntas (usuario_id, genero y
# anio_de_nacimiento ni se cargan); p3_preparar_modelado agrega la duración y la partición (3.2)
columnas = [c for c in P3_COLUMNAS_MODELADO if c != "duracion_min"]
limpios = dataset.p3_preparar_modelado(pd.read_parquet(RUTA_VIAJES_LIMPIOS, columns=columnas))

In [ ]:
# Validación de la limpieza y de la partición: se detiene si alguna regla queda sin cumplirse
dataset.p3_validar_modelado(limpios, nomenclatura)

retirados = registros_originales - len(limpios)
print(f"Registros originales : {registros_originales:,}")
print(f"Registros limpios    : {len(limpios):,}")
print(f"Retirados            : {retirados:,} ({retirados / registros_originales:.3%})")

# El dataset limpio ya lo guardó dataset.limpiar (CCDS: datos procesados)
print(f"Archivo: {RUTA_VIAJES_LIMPIOS.relative_to(PROJ_ROOT)} "
      f"({RUTA_VIAJES_LIMPIOS.stat().st_size / 1024**2:.1f} MB)")

**Observaciones de la limpieza:**
- **Registros retirados:** 59,912 de 27,086,557 (0.221 %); quedan 27,026,645. Por regla: 758 duplicados (0.003 %), 3,961 sin nomenclatura (0.015 %), 55,034 de duración cero (0.203 %) y 159 de más de 24 h (0.001 %). La duración cero cuenta 55,034 y no 55,035 como en 2.2 (`0.02-lasd-eda-p3`) porque un viaje ya había salido por una regla anterior; el log es secuencial.
- **Regla con más impacto:** la duración cero explica el 91.9 % de lo retirado. Es la única decisión con un costo apreciable, y es la que se someterá a prueba de sensibilidad en el modelado (con y sin esos viajes).
- **Validación:** las cuatro verificaciones pasan (`viaje_id` único, estaciones con nomenclatura, duración en (0, 24 h] y sin nulos en fechas ni estaciones). El dataset limpio lo guardó `dataset.limpiar` en `data/processed/viajes_limpios_2020-01_2026-09.parquet`.
- **Lo que no se hizo:** no se imputó ningún valor, no se recortaron duraciones entre 1 y 24 h y no se rellenó ningún mes (con los 81 meses no hay hueco). `usuario_id`, `genero` y `anio_de_nacimiento` no pasan al dataset de modelado: el archivo de `data/processed/` las conserva (con los años de nacimiento imposibles anulados) y este notebook las descarta al seleccionar `COLUMNAS_MODELADO`.

### 3.2 Partición temporal

La partición es **temporal**: el entrenamiento ocupa el periodo más antiguo, la validación el siguiente y la prueba el más reciente. Una partición aleatoria mezclaría días vecinos entre conjuntos y el modelo vería, durante el entrenamiento, información de los días que luego debería predecir.

| Conjunto | Periodo | Para qué se usa |
|---|---|---|
| **train** | 2020-01-01 a 2024-09-30 | Ajustar modelos y calcular todo parámetro de preprocesamiento |
| **valid** | 2024-10-01 a 2025-09-30 | Elegir hiperparámetros y umbrales |
| **test** | 2025-10-01 a 2026-09-30 | Evaluación final, una sola vez |

Los bloques de validación y prueba duran 12 meses para que cada uno cubra las cuatro estaciones del año. Las **24 horas previas a cada frontera** quedan en un conjunto `brecha` que no se usa: una etiqueta de P1 mira hasta 24 horas adelante, y sin la brecha los últimos registros de un conjunto tendrían etiquetas calculadas con datos del siguiente.

In [ ]:
# La partición la asigna dataset.p3_asignar_particion; las fronteras y la brecha están en config.py
resumen_particion = limpios.groupby("particion", observed=True).agg(
    viajes=("viaje_id", "size"),
    primer_viaje=("inicio_del_viaje", "min"),
    ultimo_viaje=("inicio_del_viaje", "max"),
)
resumen_particion["% de los viajes"] = (resumen_particion["viajes"] / len(limpios) * 100).round(1)
resumen_particion

In [ ]:
# Viajes por mes, coloreados por conjunto (el orden temporal sin superposición ya se validó arriba)
fig = plots.p3_grafica_particion(features.p3_viajes_por_mes_y_conjunto(limpios))

**Observaciones de la partición:**
- **Tamaño de los conjuntos:** train 17,987,077 viajes (66.6 %), valid 4,520,744 (16.7 %) y test 4,489,362 (16.6 %). Valid y test son casi del mismo tamaño, lo que hace comparables sus métricas.
- **Brecha:** 29,462 viajes (0.1 %) quedan fuera, las 24 horas previas a 2024-10-01 y a 2025-10-01. El costo en datos es despreciable.
- **Cobertura estacional:** valid (2024-10 a 2025-09) y test (2025-10 a 2026-09) cubren cada uno 12 meses completos, así que incluyen las cuatro estaciones del año. Train abarca 2020-01 a 2024-09 sin huecos.
- **Riesgo:** train contiene el choque de la pandemia (2020) y test no; la diferencia de régimen puede mejorar o empeorar el error por razones ajenas al modelo. Además, 2026-09 (490,056 viajes, el máximo de la serie) cae en test y sigue sin verificarse.

### 3.2.1 Entrega del dataset de modelado

Se guarda el dataset limpio con las columnas de modelado y la partición temporal. Es la entrada del baseline (`0.04-lasd-baseline-p3`), de modo que ningún notebook posterior recalcula la limpieza ni la partición.

In [ ]:
# Dataset de modelado: viajes limpios con la partición temporal (CCDS: datos procesados)
limpios.to_parquet(RUTA_VIAJES_MODELADO, index=False)
print(f"Archivo: {RUTA_VIAJES_MODELADO.relative_to(PROJ_ROOT)} "
      f"({RUTA_VIAJES_MODELADO.stat().st_size / 1024**2:.1f} MB)")
print(f"Registros: {len(limpios):,} | columnas: {list(limpios.columns)}")

### 3.3 Preprocesamiento

Se preparan dos tablas que reutilizarán los modelos:
- **Tabla de estaciones** (`estaciones_train`): capacidad, ubicación y actividad, calculada **solo con el entrenamiento**. Es la que usará P3 y la que define el nivel base de cada estación en P1.
- **Demanda por estación-hora** (`demanda_estacion_hora`): viajes que inician en cada estación y hora, con tres reglas para no inventar datos:
  1. Los ceros comienzan en la **primera hora activa** de cada estación; antes de abrir no hay demanda cero, no hay estación.
  2. Se excluyen las horas de **1:00 a 4:59**, en las que el servicio no opera.
  3. Se excluyen las horas de los meses **sin datos** (`MESES_SIN_DATOS`, calculados de los datos; ninguno con los 81 meses publicados).

In [ ]:
# Tabla de estaciones calculada solo con el conjunto de entrenamiento
estaciones_train = features.p3_tabla_estaciones(limpios, nomenclatura)

print("Estaciones en la nomenclatura          :", len(estaciones_train))
print("Con salidas en entrenamiento           :", int(estaciones_train["activa_en_train"].sum()))
print("Sin viajes en entrenamiento (nuevas)   :", int((~estaciones_train["activa_en_train"]).sum()))
print("Con capacidad 0 (fuera de P2)          :", int(estaciones_train["capacidad_cero"].sum()))
print("Con menos de 1,000 salidas en train    :", int((estaciones_train["salidas_train"] < 1000).sum()))

estaciones_train.to_parquet(RUTA_ESTACIONES_TRAIN)
estaciones_train.sort_values("salidas_train", ascending=False).head()

In [ ]:
# Demanda por estación-hora: conteo de viajes y rejilla completa desde la primera hora activa
demanda_estacion_hora, excluidos = features.p3_demanda_estacion_hora(limpios, MES_INICIO, MES_FIN)

print(f"Estaciones en la rejilla        : {demanda_estacion_hora['origen_id'].nunique()}")
print(f"Filas estación-hora             : {len(demanda_estacion_hora):,}")
print(f"Horas sin viajes (ceros)        : {(demanda_estacion_hora['viajes'] == 0).mean():.1%}")
print(f"Estación-horas con viajes fuera de la rejilla: {excluidos:,}")
print(f"Viajes totales en la rejilla    : {int(demanda_estacion_hora['viajes'].sum()):,}")
print(f"Viajes limpios                  : {len(limpios):,}")

demanda_estacion_hora.to_parquet(RUTA_DEMANDA_ESTACION_HORA, index=False)
demanda_estacion_hora.groupby("particion", observed=True)["viajes"].describe().round(2)

**Observaciones del preprocesamiento:**
- **Estaciones:** de las 484 de la nomenclatura, 363 tienen viajes en entrenamiento y 121 no: 109 aparecen después de 2024-09 (todas con viajes en test) y 12 no registran ningún viaje en todo el periodo. Cuatro tienen capacidad 0 y 122 tienen menos de 1,000 salidas en train.
- **Demanda estación-hora:** 16,179,853 filas para 472 estaciones, con 27,026,452 viajes. Se excluyeron 193 viajes (183 estación-horas) que caen en horas de 1:00 a 4:59. La rejilla arranca en la primera hora activa de cada estación, por lo que ninguna estación tiene ceros antes de existir.
- **Dispersión:** el 44.2 % de las estación-horas no tiene viajes; la mediana es 1 y la media 1.70 en train. Es una variable de conteo con muchos ceros, lo que respalda el objetivo Poisson y obliga a comparar contra el baseline en MAE, porque predecir cero ya acierta casi la mitad de las veces. Se revisará el máximo de 207 viajes en una estación-hora de validación.
- **Estaciones nuevas (hallazgo):** 109 estaciones no tienen historia en train y aparecen en test, donde generan 168,386 viajes (3.8 %). Esto baja la media de test (1.54 contra 1.70) y sube su proporción de ceros (46.5 % contra 43.8 %) sin que la demanda haya cambiado. Para ellas `demanda_media_estacion` no existe.
- **Pendiente:** definir el tratamiento de las estaciones nuevas en P1: un indicador `estacion_nueva`, un nivel base tomado de la capacidad o de la zona y una métrica que las reporte aparte. Para P3, las estaciones sin historia en train quedan fuera del agrupamiento y se asignan después al grupo más cercano.